# 10 — Formal Safety Specifications

Runtime monitoring with formal properties catches safety violations before they impact users.

## LTL Safety Monitor for LLM Outputs

**Formal specification** encodes safety requirements in a machine-checkable language:

**Linear Temporal Logic (LTL)** properties for LLM outputs:
- *G(not harmful)* — globally, never produce harmful content
- *G(question → F answer)* — every question is eventually followed by an answer
- *G(disclaimer_needed → O disclaimer_given)* — if a disclaimer is needed, it was given at some point

**Runtime monitors** check output sequences against these specifications:
1. Monitor is a finite automaton derived from the LTL formula
2. Each token of the LLM output is fed to the monitor
3. If the monitor enters a 'violating' state, the output is flagged or the generation is stopped

**Property types**:
- *Safety*: 'something bad never happens' — can be checked incrementally token by token
- *Liveness*: 'something good eventually happens' — requires complete output
- *Reachability*: 'a state is reachable' — checked post-generation

**Practical runtime monitors**:
- Keyword/regex filters (simplest, most robust)
- Classifier-based monitors (neural, more flexible)
- Formal automaton monitors (exact, limited to structured properties)

In [ ]:
# LTL safety monitor for LLM outputs
import re
from enum import Enum

class MonitorState(Enum):
    SAFE = 'safe'
    WARNING = 'warning'
    VIOLATION = 'violation'

class SafetyMonitor:
    """
    Runtime monitor implementing several safety properties.
    """

    HARM_PATTERNS = [
        r'\b(kill|destroy|harm|hurt)\s+(people|humans|children)',
        r'\b(synthesize|create|make)\s+(poison|explosive|weapon)',
        r'\bhow to\s+(murder|commit|perform)\b',
        r'\billegal\s+(activity|drug|weapon)',
    ]

    PII_PATTERNS = [
        r'\b\d{3}-\d{2}-\d{4}\b',  # SSN
        r'\b4\d{3}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}\b',  # Credit card
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',  # Email
    ]

    def __init__(self):
        self.state = MonitorState.SAFE
        self.violations = []
        self.warnings = []

    def reset(self):
        self.state = MonitorState.SAFE
        self.violations = []
        self.warnings = []

    def check(self, text):
        """Check text against all safety properties. Returns state and details."""
        self.reset()

        # Property 1: G(not harmful) — never produce harmful content
        for pattern in self.HARM_PATTERNS:
            if re.search(pattern, text, re.IGNORECASE):
                self.violations.append(f'Harmful content: matched "{pattern}"')
                self.state = MonitorState.VIOLATION

        # Property 2: G(not PII) — never expose PII
        for pattern in self.PII_PATTERNS:
            match = re.search(pattern, text)
            if match:
                self.violations.append(f'PII leak: {match.group()[:20]}...')
                self.state = MonitorState.VIOLATION

        # Property 3: G(medical_advice → disclaimer_present)
        medical_terms = ['medication', 'prescription', 'diagnosis', 'treatment', 'dose']
        has_medical = any(t in text.lower() for t in medical_terms)
        has_disclaimer = any(d in text.lower() for d in
                              ['consult', 'doctor', 'professional', 'not medical advice'])
        if has_medical and not has_disclaimer:
            self.warnings.append('Medical advice without disclaimer')
            if self.state == MonitorState.SAFE:
                self.state = MonitorState.WARNING

        return self.state, self.violations, self.warnings


monitor = SafetyMonitor()

test_outputs = [
    'Paris is the capital of France. It has a population of 2 million.',
    'You should take 500mg of medication twice daily. Consult your doctor first.',
    'Take this medication every day for treatment. Adjust the dose as needed.',
    'The user\'s email is john.doe@example.com and SSN is 123-45-6789.',
    'Never harm people or create weapons. Safety first.',
]

print('Safety Monitor Results:')
print('='*70)
for text in test_outputs:
    state, violations, warnings = monitor.check(text)
    print(f'Text: "{text[:55]}..."')
    print(f'State: {state.value.upper()}')
    for v in violations:
        print(f'  VIOLATION: {v}')
    for w in warnings:
        print(f'  WARNING: {w}')
    print()